# Smart Travel - Exploratory Data Analysis

This notebook is a mini data report for the Smart Travel recommendation system.

The goal is to ask useful product and data questions, then answer them with tables, charts, and short written insights.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "../data/processed/destinations_with_places.csv"
df = pd.read_csv(DATA_PATH)

place_count_columns = [
    "restaurant_count",
    "cafe_count",
    "bar_count",
    "museum_count",
    "park_count",
    "beach_count",
]

score_columns = [
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
]

weather_columns = [
    "winter_avg_temp",
    "winter_avg_daily_rain",
    "spring_avg_temp",
    "spring_avg_daily_rain",
    "summer_avg_temp",
    "summer_avg_daily_rain",
    "autumn_avg_temp",
    "autumn_avg_daily_rain",
]

## 1. Dataset Overview

### Question

What does the dataset contain?

### Analysis

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

### Insight

The dataset combines destination identity, geocoding, seasonal weather, OpenStreetMap place counts, and derived experience scores. This makes it suitable for both exploratory analysis and recommendation logic.

## 2. Data Quality

### Question

Are there missing or suspicious values?

### Analysis

In [ ]:
df.isnull().sum()

In [ ]:
df[place_count_columns + score_columns].describe().T

In [ ]:
df[["destination_name", "search_name", "country", "latitude", "longitude"]]

### Insight

Some destinations required a separate `search_name` because region-level coordinates were too broad. This improved POI collection for islands and regions such as Kefalonia, Mallorca, Sardinia, and Cinque Terre while keeping `destination_name` user-facing.

## 3. Destination Profiles

### Question

Which destinations stand out by food, beach, culture, nightlife, and nature?

### Analysis

In [ ]:
df.sort_values("restaurant_count", ascending=False).head(10)[[
    "destination_name", "country", "restaurant_count", "cafe_count", "food_score"
]]

In [ ]:
top_food = df.sort_values("food_score", ascending=False).head(10)

sns.barplot(data=top_food, y="destination_name", x="food_score")
plt.title("Top 10 Destinations By Food Score")
plt.xlabel("Food score")
plt.ylabel("Destination")
plt.show()

In [ ]:
top_beaches = df.sort_values("beach_count", ascending=False).head(10)
top_beaches[["destination_name", "country", "beach_count", "beach_score"]]

In [ ]:
sns.barplot(data=top_beaches, y="destination_name", x="beach_count")
plt.title("Top 10 Destinations By Beach Count")
plt.xlabel("Beach count within 3 km")
plt.ylabel("Destination")
plt.show()

In [ ]:
score_profile = df.set_index("destination_name")[score_columns]

plt.figure(figsize=(12, 8))
sns.heatmap(score_profile, cmap="viridis", linewidths=0.5)
plt.title("Destination Score Profiles")
plt.xlabel("Score")
plt.ylabel("Destination")
plt.show()

### Insight

Large cities tend to dominate food, nightlife, and culture counts, while coastal destinations stand out more clearly on beach-related features. This is an early sign that destination type matters and should be considered before modeling.

### Interactive Map

This map shows the geographic distribution of destinations. Marker size is based on `food_score`, and popups show the main experience scores.

In [ ]:
try:
    import folium
except ImportError:
    folium = None

if folium is None:
    print("folium is not installed. Install it to render the interactive map.")
else:
    travel_map = folium.Map(location=[47, 12], zoom_start=4, tiles="CartoDB positron")

    for _, row in df.iterrows():
        radius = 5 + (row["food_score"] / 100) * 15
        popup = f"""
        <b>{row['destination_name']}, {row['country']}</b><br>
        Search name: {row['search_name']}<br>
        Food score: {row['food_score']}<br>
        Nightlife score: {row['nightlife_score']}<br>
        Culture score: {row['culture_score']}<br>
        Nature score: {row['nature_score']}<br>
        Beach score: {row['beach_score']}
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            popup=folium.Popup(popup, max_width=300),
            color="#2563eb",
            fill=True,
            fill_color="#22c55e",
            fill_opacity=0.7,
        ).add_to(travel_map)

    travel_map

## 4. Relationships Between Features

### Question

Which scores or counts move together?

### Analysis

In [ ]:
correlation_columns = ["latitude", "longitude"] + weather_columns + place_count_columns + score_columns
corr = df[correlation_columns].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="viridis", center=0, annot=False, linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
upper_mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
corr_pairs = corr.where(upper_mask).stack().reset_index()
corr_pairs.columns = ["feature_1", "feature_2", "correlation"]
corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()

corr_pairs.sort_values("abs_correlation", ascending=False).head(15)

In [ ]:
target_relationships = [
    ("restaurant_count", "bar_count"),
    ("food_score", "nightlife_score"),
    ("summer_avg_temp", "beach_count"),
    ("cafe_count", "museum_count"),
]

for x_column, y_column in target_relationships:
    sns.regplot(data=df, x=x_column, y=y_column)
    plt.title(f"{x_column} vs {y_column}")
    plt.show()
    print(df[[x_column, y_column]].corr().iloc[0, 1])

### Insight

Relationships between food, nightlife, and culture are especially important because they may reflect destination density rather than only traveler experience. These relationships help decide which features should be scaled or grouped before clustering.

## 5. Outliers and Dominant Destinations

### Question

Are some cities dominating because they are larger?

### Analysis

In [ ]:
df["total_poi_count"] = df[place_count_columns].sum(axis=1)

df.sort_values("total_poi_count", ascending=False).head(10)[[
    "destination_name",
    "country",
    "total_poi_count",
    "restaurant_count",
    "cafe_count",
    "bar_count",
    "museum_count",
    "park_count",
    "beach_count",
]]

In [ ]:
top_total_poi = df.sort_values("total_poi_count", ascending=False).head(10)

sns.barplot(data=top_total_poi, y="destination_name", x="total_poi_count")
plt.title("Top 10 Destinations By Total POI Count")
plt.xlabel("Total POI count within 3 km")
plt.ylabel("Destination")
plt.show()

In [ ]:
sns.boxplot(data=df[place_count_columns], orient="h")
plt.title("Distribution Of POI Counts")
plt.xlabel("Count within 3 km")
plt.show()

### Insight

A few major cities can dominate raw POI counts. This matters because a recommendation system should not automatically treat bigger as better; it may need normalized scores, destination types, or user preference weights.

## 6. Balanced Destinations

### Question

Which destinations perform well across multiple categories?

### Analysis

In [ ]:
df["average_experience_score"] = df[score_columns].mean(axis=1)
df["minimum_experience_score"] = df[score_columns].min(axis=1)

balanced_destinations = df.sort_values(
    ["average_experience_score", "minimum_experience_score"],
    ascending=False,
).head(10)

balanced_destinations[[
    "destination_name",
    "country",
    "average_experience_score",
    "minimum_experience_score",
    *score_columns,
]]

In [ ]:
balanced_plot = balanced_destinations.set_index("destination_name")[score_columns]

balanced_plot.plot(kind="bar", figsize=(14, 7))
plt.title("Score Mix For Balanced Destinations")
plt.xlabel("Destination")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Score")
plt.show()

### Insight

Balanced destinations are especially useful for recommendation because they can satisfy several preference types at once. This is a better product signal than looking at only one category in isolation.

## 7. Recommendation Validation

### Question
Why did the recommendation engine return destinations such as Nice, Mallorca, Split, Barcelona, and Dubrovnik for an August beach-and-food trip?

### Analysis
We compare the API recommendations with the original dataset. This helps us check whether the recommendations are consistent with the underlying scores, POI counts, and summer weather.


In [ ]:
recommendations = pd.read_csv("../data/processed/recommendation_sample.csv")
recommendations


In [ ]:
recommended_names = recommendations["destination_name"].tolist()

recommended_destinations = df[df["destination_name"].isin(recommended_names)][[
    "destination_name",
    "country",
    "restaurant_count",
    "cafe_count",
    "bar_count",
    "museum_count",
    "park_count",
    "beach_count",
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
    "summer_avg_temp",
    "summer_avg_daily_rain",
]].merge(
    recommendations[[
        "destination_name",
        "recommendation_score",
        "preference_score",
        "weather_score",
        "cluster_profile",
        "explanation",
    ]],
    on="destination_name",
    how="left",
)

recommended_destinations.sort_values("recommendation_score", ascending=False)


In [ ]:
score_comparison = recommended_destinations.set_index("destination_name")[[
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
]]

score_comparison.plot(kind="bar", figsize=(14, 7))
plt.title("Experience Scores For Recommended Destinations")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
weather_validation = recommended_destinations[[
    "destination_name",
    "recommendation_score",
    "summer_avg_temp",
    "summer_avg_daily_rain",
    "weather_score",
]].sort_values("recommendation_score", ascending=False)

weather_validation


### Explainable Recommendation Output

The API now returns both a natural-language explanation and a structured `reason` object. This helps us verify that the recommendation is not a black box.


In [ ]:
recommendations[[
    "destination_name",
    "recommendation_score",
    "natural_reason",
    "reason",
]]


In [ ]:
recommended_summary = recommended_destinations[[
    "food_score",
    "beach_score",
    "nature_score",
    "summer_avg_temp",
    "summer_avg_daily_rain",
]].mean()

dataset_summary = df[[
    "food_score",
    "beach_score",
    "nature_score",
    "summer_avg_temp",
    "summer_avg_daily_rain",
]].mean()

pd.DataFrame({
    "recommended_average": recommended_summary,
    "dataset_average": dataset_summary,
    "difference": recommended_summary - dataset_summary,
}).round(2)


### Insight
For an August trip with high `beach` and `food` preferences, the recommended destinations should have strong beach signals, good summer weather, and enough food-related POIs to match the user's preferences.

This validation step is important because the API returning a result is not enough. We also need to check whether the recommendations make sense from a travel perspective.


## 8. Key Insights

### Question

What did we learn from the EDA?

### Insight

1. `search_name` is important for data quality. It fixes broad region geocoding while preserving the user-facing destination name.
2. Large cities dominate raw POI counts, especially food, nightlife, and culture features.
3. Beach-oriented destinations are easier to identify through `beach_count` and `beach_score` than through raw city-density features.
4. Some features move together because they measure destination density, not just experience quality.
5. Balanced destination scores are a strong candidate for future recommendation logic.
6. The next modeling step should be K-Means clustering on scaled score and POI features to see whether natural destination groups appear.
